In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from wildlife_datasets.datasets import TurtlewatchEgypt_New, WildlifeDataset
from wildlife_datasets.detection import run_detection

from animal_re_id.config import load_paths, resolve_path
from animal_re_id.models import get_pipeline_aliked, turtle_detector, visualise_matches

In [ ]:
identity = "kiki"
years = ["2019 PICS + DATASHEET", "2023 PICS + online database"]
device = "cuda"

local_path = "../../config/paths.json"
paths = load_paths("TurtlewatchEgypt", local_path=local_path)
file_name = resolve_path(paths["file_name_individuals"])
root_new = resolve_path(paths["root_new"])

In [ ]:
model = turtle_detector()

dfs = []
for year in years:
    dataset = TurtlewatchEgypt_New(f"{root_new}/{year}", file_name=file_name)
    dataset = dataset.get_subset(dataset.metadata["identity"] == identity)
    segmentation = run_detection(dataset, model)
    df = segmentation.merge(dataset.metadata[["image_id", "identity", "path", "date"]], on="image_id")
    df["path"] = year + "/" + df["path"]
    df["year"] = int(year[:4])
    dfs.append(df)
df = pd.concat(dfs).reset_index(drop=True)
df["image_id"] = df["image_id"].astype(str) + "_" + df.index.astype(str)
df["bbox"] = df[["bbox_x", "bbox_y", "bbox_w", "bbox_h"]].to_numpy().tolist()
df = df.drop(columns=["segmentation"])

In [ ]:
dataset = WildlifeDataset(root=root_new, df=df, img_load="bbox", load_label=True)
aliked = get_pipeline_aliked(device=device, cache_prefix="test_cache")

In [ ]:
labels = ["head", "flipper_fl", "flipper_fr", "flipper_hl", "flipper_hr"]
year1 = int(years[0][:4])
year2 = int(years[1][:4])

metadata = dataset.metadata.reset_index(drop=True)
year = metadata["year"].to_numpy()
label = metadata["label"].to_numpy()

same_label = label[:, None] == label[None, :]
cross_year = (year[:, None] == year1) & (year[None, :] == year2)
pairs = np.argwhere(same_label & cross_year)

similarity = aliked(dataset, dataset, pairs=pairs)
similarity = np.maximum(np.nan_to_num(similarity, nan=-1), np.nan_to_num(similarity.T, nan=-1))

In [ ]:
parts = {
    "head": ["head"],
    "front flipper": ["flipper_fl", "flipper_fr"],
    "hind flipper": ["flipper_hl", "flipper_hr"],
}

paths = metadata["path"].to_numpy()
paths1 = np.unique(paths[year == year1])
paths2 = np.unique(paths[year == year2])

results = []
for path1 in paths1:
    for path2 in paths2:
        mask1 = paths == path1
        mask2 = paths == path2
        row = {"path1": path1, "path2": path2}
        for part, part_labels in parts.items():
            part_mask = np.isin(label, part_labels)
            idx1 = np.where(mask1 & part_mask)[0]
            idx2 = np.where(mask2 & part_mask)[0]
            if len(idx1) == 0 or len(idx2) == 0:
                row[part] = 0
            else:
                row[part] = similarity[np.ix_(idx1, idx2)].max()
        results.append(row)

results = pd.DataFrame(results)
results["min"] = results[list(parts)].min(axis=1)
results = results.sort_values("min", ascending=False)
results.head(10)

In [ ]:
top = results.iloc[2]

for path in [top["path1"], top["path2"]]:
    result = model(f"{root_new}/{path}", verbose=False)[0]
    names = np.array([result.names[int(c)] for c in result.boxes.cls])
    keep = np.where(names != "turtle")[0]
    result = result[keep]

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(result.orig_img[:, :, ::-1])
    ax.axis("off")

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(result.plot()[:, :, ::-1])
    ax.axis("off")

In [ ]:
mask1 = paths == top["path1"]
mask2 = paths == top["path2"]
for part, part_labels in parts.items():
    part_mask = np.isin(label, part_labels)
    idx1 = np.where(mask1 & part_mask)[0]
    idx2 = np.where(mask2 & part_mask)[0]
    sub = similarity[np.ix_(idx1, idx2)]
    i, j = np.unravel_index(sub.argmax(), sub.shape)
    row = {
        "image_id1": metadata.loc[idx1[i], "image_id"],
        "image_id2": metadata.loc[idx2[j], "image_id"],
    }
    fig, ax = plt.subplots(figsize=(10, 5))
    title = f"{part}: {sub[i, j]:.0f} matches"
    visualise_matches(dataset, dataset, row, aliked, "aliked", ax=ax, title=title)